# Sensitivity Analysis for CineInfini Parametric Cost Model

This notebook demonstrates how to vary parameters and observe the impact on total production cost (mode IA).

First, ensure that `cineinfini_cost_model_param.py` is in the same directory or in your Python path.

In [ ]:
import sys
sys.path.append('..')  # if notebook is in a subfolder; adjust as needed
from src.cineinfini_cost_model_param import compute_cost_ia, DEFAULT_PARAMS

## 1. Baseline costs (low, medium, high scenarios)

In [ ]:
costs = compute_cost_ia(scenario='all')
print("Baseline costs (90 min film):")
for k, v in costs.items():
    print(f"  {k.capitalize():6s} : {v:.2f} {DEFAULT_PARAMS['currency']}")
print()

## 2. Vary film duration

In [ ]:
for duration in [3600, 5400, 7200, 10800]:  # 60, 90, 120, 180 minutes
    params = {"film_duration_sec": duration}
    cost = compute_cost_ia(scenario='medium', user_params=params)
    print(f"Duration {duration//60} min: {cost:.2f} {DEFAULT_PARAMS['currency']}")

## 3. Vary rejection rate (requires overriding constants)

We need to temporarily modify the `INTERVALS` dictionary.

In [ ]:
from src.cineinfini_cost_model_param import INTERVALS, compute_cost_ia

def cost_with_rejection_rate(rej):
    custom_intervals = INTERVALS.copy()
    custom_intervals["REJECTION_RATE"] = (rej, rej, rej)
    import src.cineinfini_cost_model_param as cm
    original = cm.INTERVALS
    cm.INTERVALS = custom_intervals
    cost = compute_cost_ia(scenario='medium')
    cm.INTERVALS = original
    return cost

for rej in [0.2, 0.5, 1.0, 2.0]:
    c = cost_with_rejection_rate(rej)
    print(f"Rejection rate {rej}: {c:.2f} {DEFAULT_PARAMS['currency']}")

## 4. Visualisation (requires matplotlib)

In [ ]:
import matplotlib.pyplot as plt

scenarios = ['low', 'medium', 'high']
values = [compute_cost_ia(s) for s in scenarios]

plt.bar(scenarios, values, color=['green', 'orange', 'red'])
plt.ylabel(f'Production cost ({DEFAULT_PARAMS["currency"]})')
plt.title('CineInfini cost estimates (90 min AI film)')
plt.show()